   %md
%md
# Notebook 13: CMS DME Anomaly Detection (Referring Providers)

**Question:** Which prescribers order durable medical equipment at volumes far outside their peers, and what does that reveal about the start of the DME handoff?

**Data:** Medicare Durable Medical Equipment, Devices & Supplies by Referring Provider and Service, 2024 (CMS). Original Medicare Part B only; Medicare Advantage patients are not included; small counts suppressed for privacy.
Source: https://data.cms.gov/provider-summary-by-type-of-service/medicare-durable-medical-equipment-devices-supplies/medicare-durable-medical-equipment-devices-supplies-by-referring-provider-and-service

**Publishing rule:** No individual provider names or identifiers in any output. Results are reported by specialty and equipment type only. An outlier flag is a statistical signal, not evidence of wrongdoing.

In [0]:
import urllib.request, os

URL = "https://data.cms.gov/sites/default/files/2026-07/1ae6eac7-c8a5-4f1c-a82e-617f4bb65a5f/mup_dme_ry26_p05_v10_dy24_rfrhpr.csv"
spark.sql("CREATE VOLUME IF NOT EXISTS workspace.default.raw_data")
DEST = "/Volumes/workspace/default/raw_data/dme_referring_provider_service_2024.csv"

if not os.path.exists(DEST):
    urllib.request.urlretrieve(URL, DEST)
print(f"Saved: {os.path.getsize(DEST)/1e6:,.0f} MB")

In [0]:
raw = (spark.read.option("header", True).option("inferSchema", True)
       .option("quote", '"').option("escape", '"').csv(DEST))
raw.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.default.dme_referring_2024_bronze")

df = spark.table("workspace.default.dme_referring_2024_bronze")
print(f"{df.count():,} rows, {len(df.columns)} columns")
df.printSchema()

In [0]:
from pyspark.sql import functions as F

for c in ["Tot_Suplrs", "Tot_Suplr_Benes"]:
    nulls = df.filter(F.col(c).isNull()).count()
    non_numeric = df.filter(F.col(c).isNotNull() & F.col(c).cast("int").isNull())
    print(f"{c}: {nulls:,} blank rows | {non_numeric.count():,} non-numeric rows")
    display(non_numeric.groupBy(c).count().orderBy(F.desc("count")).limit(10))

In [0]:
for c in ["Tot_Suplr_Clms", "Tot_Suplr_Srvcs", "Avg_Suplr_Mdcr_Stdzd_Amt"]:
    print(f"{c}: {df.filter(F.col(c).isNull()).count():,} blank rows")

print("\nLongest values in Tot_Suplrs:")
display(df.select("Tot_Suplrs", F.length("Tot_Suplrs").alias("len")).distinct().orderBy(F.desc("len")).limit(10))

In [0]:
bad = df.filter(~F.col("Tot_Suplrs").rlike("^[0-9]+$"))
print(f"Malformed rows: {bad.count():,}")
display(bad.select("HCPCS_CD", "Tot_Suplrs", "Tot_Suplr_Clms", "Tot_Suplr_Srvcs").limit(10))

In [0]:
raw_fixed = (spark.read.option("header", True).option("inferSchema", True)
             .option("quote", '"').option("escape", '"').csv(DEST))
print(f"{raw_fixed.count():,} rows")
print(dict(raw_fixed.dtypes)["Tot_Suplrs"], dict(raw_fixed.dtypes)["Tot_Suplr_Benes"])

In [0]:
raw_fixed.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.default.dme_referring_2024_bronze")
df = spark.table("workspace.default.dme_referring_2024_bronze")

for c in ["Tot_Suplrs", "Tot_Suplr_Benes", "Tot_Suplr_Clms", "Tot_Suplr_Srvcs"]:
    print(f"{c}: {df.filter(F.col(c).isNull()).count():,} blank rows")
print(f"K0056 rows: {df.filter(F.col('HCPCS_CD') == 'K0056').count():,}")

In [0]:
silver = (df
  .withColumn("Rfrg_NPI", F.col("Rfrg_NPI").cast("string"))
  .select("Rfrg_NPI", "Rfrg_Prvdr_State_Abrvtn", "Rfrg_Prvdr_RUCA_Cat", "Rfrg_Prvdr_Spclty_Desc",
          "RBCS_Desc", "HCPCS_CD", "HCPCS_Desc", "Suplr_Rentl_Ind",
          "Tot_Suplrs", "Tot_Suplr_Benes", "Tot_Suplr_Clms", "Tot_Suplr_Srvcs",
          "Avg_Suplr_Mdcr_Stdzd_Amt"))

silver.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.default.dme_referring_2024_silver")
print(f"Silver: {silver.count():,} rows, {len(silver.columns)} columns (names and addresses removed)")

In [0]:
print("By equipment category:")
display(silver.groupBy("RBCS_Desc")
        .agg(F.countDistinct("Rfrg_NPI").alias("prescribers"),
             F.sum("Tot_Suplr_Clms").alias("claims"))
        .orderBy(F.desc("claims")))

print("Top 15 prescribing specialties:")
display(silver.groupBy("Rfrg_Prvdr_Spclty_Desc")
        .agg(F.countDistinct("Rfrg_NPI").alias("prescribers"),
             F.sum("Tot_Suplr_Clms").alias("claims"))
        .orderBy(F.desc("claims")).limit(15))

In [0]:
display(silver.filter(F.col("RBCS_Desc") == "DME-Other DME")
        .groupBy("HCPCS_CD", "HCPCS_Desc")
        .agg(F.countDistinct("Rfrg_NPI").alias("prescribers"),
             F.sum("Tot_Suplr_Clms").alias("claims"))
        .orderBy(F.desc("claims")).limit(15))

In [0]:
CPAP_CODES = ["E0601", "E0562", "A4604", "A7030", "A7031", "A7032", "A7033", "A7034",
              "A7035", "A7036", "A7037", "A7038", "A7039", "A7046"]
cpap = silver.filter(F.col("HCPCS_CD").isin(CPAP_CODES))

coverage = (cpap.groupBy("HCPCS_CD", "HCPCS_Desc")
    .agg(F.count("*").alias("rows"),
         F.sum(F.when(F.col("Tot_Suplr_Benes").isNotNull(), 1).otherwise(0)).alias("rows_visible"),
         F.sum("Tot_Suplr_Srvcs").alias("services"),
         F.sum(F.when(F.col("Tot_Suplr_Benes").isNotNull(), F.col("Tot_Suplr_Srvcs")).otherwise(0)).alias("services_visible"))
    .withColumn("pct_rows_visible", F.round(F.col("rows_visible") / F.col("rows") * 100, 1))
    .withColumn("pct_services_visible", F.round(F.col("services_visible") / F.col("services") * 100, 1))
    .orderBy("HCPCS_CD"))
display(coverage)

In [0]:
ANNUAL_MAX = {"A7030": 4, "A7031": 12, "A7032": 24, "A7033": 24, "A7034": 4, "A7035": 2,
              "A7036": 2, "A7037": 4, "A7038": 24, "A7039": 2, "A7046": 2, "A4604": 4}
limits = spark.createDataFrame(list(ANNUAL_MAX.items()), ["HCPCS_CD", "annual_max_per_patient"])

supply = (cpap.filter(F.col("Tot_Suplr_Benes").isNotNull())
          .join(limits, "HCPCS_CD")
          .withColumn("srv_per_patient", F.col("Tot_Suplr_Srvcs") / F.col("Tot_Suplr_Benes"))
          .withColumn("pct_of_max", F.col("srv_per_patient") / F.col("annual_max_per_patient")))

summary = (supply.groupBy("HCPCS_CD", "annual_max_per_patient")
    .agg(F.count("*").alias("prescriber_rows"),
         F.round(F.expr("percentile_approx(srv_per_patient, 0.5)"), 2).alias("median_per_patient"),
         F.round(F.expr("percentile_approx(srv_per_patient, 0.95)"), 2).alias("p95_per_patient"),
         F.sum(F.when(F.col("pct_of_max") > 1, 1).otherwise(0)).alias("rows_above_max"))
    .withColumn("pct_rows_above_max", F.round(F.col("rows_above_max") / F.col("prescriber_rows") * 100, 1))
    .orderBy("HCPCS_CD"))
display(summary)

In [0]:
print("Rows with zero allowed amount:", df.filter(F.col("Avg_Suplr_Mdcr_Alowd_Amt") == 0).count())
display(df.filter(F.col("HCPCS_CD").isin(CPAP_CODES))
          .agg(F.min("Avg_Suplr_Mdcr_Alowd_Amt").alias("min_allowed"),
               F.expr("percentile_approx(Avg_Suplr_Mdcr_Alowd_Amt, 0.01)").alias("p1_allowed")))

In [0]:
from pyspark.sql import Window

peer = Window.partitionBy("HCPCS_CD", "Rfrg_Prvdr_Spclty_Desc")

scored = (supply
    .withColumn("peer_n", F.count("*").over(peer))
    .withColumn("peer_median", F.percentile_approx("srv_per_patient", 0.5).over(peer))
    .withColumn("abs_dev", F.abs(F.col("srv_per_patient") - F.col("peer_median")))
    .withColumn("peer_mad", F.percentile_approx("abs_dev", 0.5).over(peer))
    .filter(F.col("peer_n") >= 30)
    .withColumn("robust_z", F.when(F.col("peer_mad") > 0,
                0.6745 * (F.col("srv_per_patient") - F.col("peer_median")) / F.col("peer_mad"))))

scored.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.default.dme_cpap_peer_scores_gold")

flags = (scored.groupBy("HCPCS_CD")
    .agg(F.count("*").alias("rows_scored"),
         F.sum(F.when(F.col("robust_z") > 3.5, 1).otherwise(0)).alias("high_outliers"),
         F.sum(F.when(F.col("robust_z") < -3.5, 1).otherwise(0)).alias("low_outliers"))
    .withColumn("pct_high", F.round(F.col("high_outliers") / F.col("rows_scored") * 100, 2))
    .withColumn("pct_low", F.round(F.col("low_outliers") / F.col("rows_scored") * 100, 2))
    .orderBy(F.desc("pct_high")))
display(flags)

In [0]:
flagged = (scored.filter(F.abs(F.col("robust_z")) > 3.5)
           .withColumn("direction", F.when(F.col("robust_z") > 0, "high").otherwise("low")))

display(flagged.groupBy("HCPCS_CD", "direction")
        .agg(F.count("*").alias("rows"),
             F.round(F.avg("peer_median"), 2).alias("peer_median"),
             F.round(F.avg("srv_per_patient"), 2).alias("flagged_avg_per_patient"),
             F.round(F.avg("pct_of_max") * 100, 0).alias("pct_of_policy_max"))
        .orderBy("HCPCS_CD", "direction"))

In [0]:
from pyspark.sql import functions as F

scored = spark.table("workspace.default.dme_cpap_peer_scores_gold")
flagged = (scored.filter(F.abs(F.col("robust_z")) > 3.5)
           .withColumn("direction", F.when(F.col("robust_z") > 0, "high").otherwise("low")))

per_npi = (flagged.filter(F.col("direction") == "high")
           .groupBy("Rfrg_NPI").agg(F.countDistinct("HCPCS_CD").alias("items_flagged_high")))
display(per_npi.groupBy("items_flagged_high").count().orderBy("items_flagged_high"))

In [0]:
high_npis = flagged.filter(F.col("direction") == "high").select("Rfrg_NPI").distinct()
items_per = (scored.join(high_npis, "Rfrg_NPI")
             .groupBy("Rfrg_NPI").agg(F.countDistinct("HCPCS_CD").alias("items_ordered")))
display(items_per.agg(F.expr("percentile_approx(items_ordered, 0.5)").alias("median_items_ordered"),
                      F.min("items_ordered").alias("min"), F.max("items_ordered").alias("max")))

In [0]:
from pyspark.sql import functions as F

scored = spark.table("workspace.default.dme_cpap_peer_scores_gold")
flagged = (scored.filter(F.abs(F.col("robust_z")) > 3.5)
           .withColumn("direction", F.when(F.col("robust_z") > 0, "high").otherwise("low")))

# Check 1: how many CPAP items do high-outlier prescribers have rows for?
high_npis = flagged.filter(F.col("direction") == "high").select("Rfrg_NPI").distinct()
items_per = (scored.join(high_npis, "Rfrg_NPI")
             .groupBy("Rfrg_NPI").agg(F.countDistinct("HCPCS_CD").alias("items_ordered")))
display(items_per.agg(F.expr("percentile_approx(items_ordered, 0.5)").alias("median_items_ordered"),
                      F.min("items_ordered").alias("min"), F.max("items_ordered").alias("max")))

# Check 2: are flagged rows mostly small patient panels?
display(scored.withColumn("group", F.when(F.abs(F.col("robust_z")) > 3.5, "flagged").otherwise("not flagged"))
        .groupBy("group")
        .agg(F.count("*").alias("rows"),
             F.expr("percentile_approx(Tot_Suplr_Benes, 0.5)").alias("median_patients"),
             F.expr("percentile_approx(Tot_Suplr_Benes, 0.25)").alias("p25_patients")))

In [0]:
robust = flagged.filter(F.col("Tot_Suplr_Benes") >= 20)
display(robust.groupBy("direction").count())
print(f"Flags surviving a 20-patient minimum: {robust.count()} of {flagged.count()}")

In [0]:
from pyspark.sql import functions as F

CPAP_CODES = ["E0601", "E0562", "A4604", "A7030", "A7031", "A7032", "A7033", "A7034",
              "A7035", "A7036", "A7037", "A7038", "A7039", "A7046"]

robust = (spark.table("workspace.default.dme_cpap_peer_scores_gold")
          .filter((F.abs(F.col("robust_z")) > 3.5) & (F.col("Tot_Suplr_Benes") >= 20))
          .withColumn("direction", F.when(F.col("robust_z") > 0, "high").otherwise("low")))
display(robust.groupBy("HCPCS_CD", "direction").count().orderBy("direction", "HCPCS_CD"))

display(spark.table("workspace.default.dme_referring_2024_bronze")
        .filter(F.col("HCPCS_CD").isin(CPAP_CODES))
        .groupBy("HCPCS_CD", "HCPCS_Desc")
        .agg(F.round(F.expr("percentile_approx(Avg_Suplr_Mdcr_Alowd_Amt, 0.5)"), 2).alias("median_allowed_per_unit"))
        .orderBy("median_allowed_per_unit"))

%md
## Findings: Notebook 13

**Data:** CMS Medicare DME by Referring Provider and Service, 2024: 1,347,255 prescriber × item × rental rows. Original Medicare only. Rows with 10 or fewer claims are excluded, and patient counts are blank in 69% of rows.

**Data quality catch:** The default CSV reader shifted columns for one wheelchair code (K0056) because its description contained commas and doubled quotes. Fixed with `escape='"'`, and a guardrail now stops the notebook if it recurs.

**CPAP dominates DME volume:** CPAP devices and supplies account for about 23 million claims in the top 15 "Other DME" codes, roughly 40% of all DME claims. Most of that is recurring resupply, not devices.

**Against policy limits:** No prescriber row averaged above Medicare's replacement limits (LCD L33718). This is likely by construction: counts reflect allowed services (0 rows with a $0 allowed amount), so over-limit units that were denied do not appear. Median resupply ran at about **half** of the policy allowance.

**Against peers (robust z-score > 3.5 within item × specialty):** 173 of 160,582 rows were flagged (0.11%). Flagged rows had a median of 13 patients vs. 24 for all others. With a 20-patient minimum, only **27 flags survived: 7 high and 20 low.** 17 of the 20 low flags were on items under $20 per unit, 10 of them on disposable filters.

**Template hypothesis:** Not supported for most outliers. 80 of 88 high-outlier prescribers were flagged on only one item, despite ordering a median of 5 CPAP items.

**What I'd tell a COO:** Allowed CPAP resupply runs at about half of what Medicare permits, and apparent over-ordering was mostly small-sample noise. The signal that holds up is under-supply of low-cost items, mainly filters, which may point to refill follow-up gaps or patient education, since these are the items least likely to get attention.

**What this data can't answer:** Whether over-supply was attempted, because denied units are not shown. And why some patients receive so few supplies: stopped therapy, late start, or no refill outreach all look the same in averages.

**What I'd need next:** Denied-claim data for the over-supply question. For under-supply: patient-level resupply dates, CPAP usage data, supplier refill-contact records, and the supplier linked to each patient.

**Judgment calls:** Peer groups of 30 or more; a robust z-score cutoff of 3.5; a 20-patient minimum. **Publishing rule:** no provider names or identifiers in any output.